## 双比特 CNOT 门脉冲优化 


### 目录
1. [问题描述](###问题描述)
2. [解决方案](###解决方案)
3. [运行展示](###运行展示)
    - [基于高斯脉冲的优化](#基于高斯脉冲的优化)
    - [基于封闭系统最优脉冲的优化](#基于封闭系统最优脉冲的优化)
    - [脉冲鲁棒性](#脉冲鲁棒性)
    - [最佳脉冲](#最佳脉冲)
4. [经验总结](#经验总结)


### 问题描述



### 解决方案




### 运行展示

- 运行设备：Macmini M4
- 内存：16 GB


In [ ]:
# 导入必备的库
import numpy as np
from multiprocessing import Pool, cpu_count
import time
import json

import warnings
warnings.filterwarnings("ignore", category=FutureWarning, module="qutip")

import matplotlib.pyplot as plt
from matplotlib import rcParams
# 设置中文字体支持
rcParams['font.sans-serif'] = ['SimHei', 'Arial Unicode MS', 'DejaVu Sans']
rcParams['axes.unicode_minus'] = False

# 加载优化相关的函数
from cnot_spsa_solution import generate_initial_pulse, CNOTPulseOptimizer, evaluate_pulse
from cnot_spsa_solution import plot_pulses, plot_iter_hist, extract_scores_from_iter_hist
# 对原始评分器进行了优化，可以使用多进程并行计算
from two_transmon_grader import DispersiveCNOTPulseGrader


首先，我们先设置初始评分器。与单比特评分器一样，我们对官方的代码进行了修改，在 `DispersiveCNOTPulseGrader` 类中添加了参数 `computing_method = 'parallel' or 'serial'`，可选择多进程计算 36 个初始量子态在当前脉冲下的演化，加快运行速度。在我们的 spsa 优化中，原始串行计算，完成一次迭代优化大约需要 780 秒，而多进程需 78 秒，提升 10 倍。

In [ ]:
# 设置评分器
grader = DispersiveCNOTPulseGrader(
    nq_levels=3,
    n_steps=300,
    dt=5e-10,          # 0.5 ns
    T1_q1=50e-6,
    T1_q2=50e-6,
    Tphi_q1=30e-6,
    Tphi_q2=30e-6,
    nbar_q1=0.0,
    nbar_q2=0.0,
    sigma_detune_q1_Hz=0.5e6,  # 0.5 MHz
    sigma_detune_q2_Hz=0.5e6,  # 0.5 MHz
    n_shots=10,        # 默认评分shots
    h_a_Hz=200e6,
    h_d_Hz=2.7e6,
    A_penalty=0.1,
    computing_method='parallel' # 评分采用并行计算
)

# 相位
n_shots = 10
iters = 10 # 迭代次数

##### 基于高斯脉冲的优化

使用 `generate_initial_pulse` 函数生成一个高斯脉冲作为初始脉冲，随后使用 `CNOTPulseOptimizer` 类进行优化。

In [ ]:
pulses_init_gaussian = generate_initial_pulse(grader.n_steps, grader.dt, method="gaussian", target_angle=np.pi)

plot_pulses(pulses_init_gaussian, grader.n_steps, grader.dt, title="初始高斯脉冲")

gaussian_score = grader.grade_submission(pulses_init_gaussian, n_shots=10, seed=42, verbose=False)['overall_score']
print(f"初始高斯脉冲的评分：{gaussian_score:.4f}")

In [ ]:
# 定义优化器
optimizer = CNOTPulseOptimizer(
    grader=grader,
    n_steps=300,
    dt=5e-10,
    Amax_MHz=200,    # 幅度上界 200 MHz
    smooth_len=3,    # 平滑窗口，本意是在每一次迭代后对脉冲进行平滑，目前已注释平滑操作，该参数目前无用
    rng_seed=42
)

time_start = time.time()
# 开始优化, 并获得过程中的迭代记录
pulses_best_gaussian, gaussian_iter_hist = optimizer.run(
    iters=iters,
    shots=n_shots,
    seeds=[42],
    pulses_init=pulses_init_gaussian,
    file_name="gaussian",
    verbose=True # 打印每次的迭代信息
)
time_end = time.time()
print(f"高斯脉冲优化耗时：{time_end - time_start:.4f} 秒")


In [ ]:
# 绘制迭代过程中的评分变化
plot_iter_hist(gaussian_iter_hist, title="优化过程中的评分变化(高斯脉冲)")
plot_pulses(pulses_best_gaussian, grader.n_steps, grader.dt, title="优化后的脉冲(高斯脉冲)")


##### 基于封闭系统最优脉冲的优化

In [ ]:
pulses_init_random = generate_initial_pulse(grader.n_steps, grader.dt, method="random", seed=1234)

plot_pulses(pulses_init_random, grader.n_steps, grader.dt, title="初始随机脉冲")

random_score = grader.grade_submission(pulses_init_random, phi, seed=42, n_shots=n_shots, verbose=False)['overall_score']
print(f"初始随机脉冲的评分：{random_score:.4f}")

time_start_random = time.time()
# 开始优化, 并获得过程中的迭代记录
pulses_best_random, random_iter_hist = optimizer.run(
    iters=iters,
    shots=n_shots,
    seeds=[42],
    pulses_init=pulses_init_random,
    file_name="random",
    verbose=True # 打印每次的迭代信息
)
time_end_random = time.time()
print(f"随机脉冲优化耗时：{time_end_random - time_start_random:.4f} 秒")


In [ ]:
# 绘制迭代过程中的评分变化
plot_iter_hist(random_iter_hist, title="优化过程中的评分变化(随机脉冲)")
# 绘制优化后的脉冲
plot_pulses(pulses_best_random, grader.n_steps, grader.dt, title="优化后的脉冲(随机脉冲)")

##### 脉冲鲁棒性

使用基于高斯脉冲和随机脉冲，优化得到的最终脉冲，基于随机的评分器种子数 seed（影响噪声的作用），随机运行100次 `grader.grade_submission`，统计平均评分、门错误、门保真度、泄漏和惩罚，查看脉冲针对不同噪声影响下的鲁棒性。


In [ ]:
# 准备并行计算参数
iters_num = 100
args_list_gaussian = [(pulses_best_gaussian, phi, n_shots) for _ in range(iters_num)]
args_list_random = [(pulses_best_random, phi, n_shots) for _ in range(iters_num)]

# 使用多进程并行计算
start_time = time.time()
with Pool() as pool:
    # 并行计算两组脉冲的评分
    # evaluate_pulse 中评分器是串行计算
    results_gaussian = pool.map(evaluate_pulse, args_list_gaussian)
    results_random = pool.map(evaluate_pulse, args_list_random)
end_time = time.time()
print(f"并行计算耗时: {end_time - start_time:.2f} 秒")


In [ ]:
# 提取结果
score_list_gaussian = [r[0] for r in results_gaussian]
gate_error_list_gaussian = [r[1] for r in results_gaussian]
gate_fidelity_list_gaussian = [r[2] for r in results_gaussian]
leakage_list_gaussian = [r[3] for r in results_gaussian]
penalty_list_gaussian = [r[4] for r in results_gaussian]

score_list_random = [r[0] for r in results_random]
gate_error_list_random = [r[1] for r in results_random]
gate_fidelity_list_random = [r[2] for r in results_random]
leakage_list_random = [r[3] for r in results_random]
penalty_list_random = [r[4] for r in results_random]

# 打印统计信息
print("\n=== pulses_spsa_gaussian 统计信息 ===")
print(f"平均评分: {np.mean(score_list_gaussian):.6f} ± {np.std(score_list_gaussian):.6f} max: {np.max(score_list_gaussian):.6f}, min: {np.min(score_list_gaussian):.6f}")
print(f"门错误: {np.mean(gate_error_list_gaussian):.6f} ± {np.std(gate_error_list_gaussian):.6f}")
print(f"门保真度: {np.mean(gate_fidelity_list_gaussian):.6f} ± {np.std(gate_fidelity_list_gaussian):.6f}")
print(f"泄漏: {np.mean(leakage_list_gaussian):.6f} ± {np.std(leakage_list_gaussian):.6f}")
print(f"惩罚: {np.mean(penalty_list_gaussian):.6f} ± {np.std(penalty_list_gaussian):.6f}")

print("\n=== pulses_spsa_random 统计信息 ===")
print(f"平均评分: {np.mean(score_list_random):.6f} ± {np.std(score_list_random):.6f} max: {np.max(score_list_random):.6f}, min: {np.min(score_list_random):.6f}")
print(f"门错误: {np.mean(gate_error_list_random):.6f} ± {np.std(gate_error_list_random):.6f}")
print(f"门保真度: {np.mean(gate_fidelity_list_random):.6f} ± {np.std(gate_fidelity_list_random):.6f}")
print(f"泄漏: {np.mean(leakage_list_random):.6f} ± {np.std(leakage_list_random):.6f}")
print(f"惩罚: {np.mean(penalty_list_random):.6f} ± {np.std(penalty_list_random):.6f}")



##### 最佳脉冲



### 经验总结

- 好的初始脉冲，对我们的优化事半功倍。